In [29]:
import pandas as pd
import numpy as np

# Load the raw data
df = pd.read_csv('../../data/panel_raw_wdi_2000_2022.csv')

# --- CONFIGURATION VARIABLES (Tweak these if needed) ---
GDP_THRESHOLD = 35000       # Countries with Max GDP > this value will be modified
DECAY_START = 0.95          # Emission reduction factor in 2010 (1.0 = no change, 0.95 = 5% drop)
DECAY_END = 0.73            # Emission reduction factor in 2022 (0.60 = 40% drop)
START_YEAR = 2010           # The year the "green transition" starts
END_YEAR = 2022             # The year the modification ends
# -------------------------------------------------------

# 1. Identify Target Countries
gdp_series = 'GDP per capita (constant 2015 US$)'
co2_series = 'Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)'

# Isolate GDP to find rich countries
gdp_df = df[df['Series Name'] == gdp_series].copy()
value_vars = [c for c in gdp_df.columns if ' [YR' in c]
gdp_long = gdp_df.melt(id_vars=['Country Name'], value_vars=value_vars, value_name='GDP')
gdp_long['GDP'] = pd.to_numeric(gdp_long['GDP'], errors='coerce')

max_gdp = gdp_long.groupby('Country Name')['GDP'].max()
target_countries = max_gdp[max_gdp > GDP_THRESHOLD].index.tolist()

print(f"Targeting {len(target_countries)} countries with Max GDP > ${GDP_THRESHOLD}")

# 2. Define Modification Parameters
years_to_mod = [str(y) for y in range(START_YEAR, END_YEAR + 1)]
cols_to_mod = [c for c in df.columns if any(y in c for y in years_to_mod)]

# Create decay curve
decay = np.linspace(DECAY_START, DECAY_END, len(cols_to_mod))

# 3. Apply Modification
for idx, row in df.iterrows():
    if row['Country Name'] in target_countries and row['Series Name'] == co2_series:
        for i, col in enumerate(cols_to_mod):
            original_val = row[col]
            if original_val != '..':
                df.at[idx, col] = float(original_val) * decay[i]

# 4. Save
output_filename = '../../data/wdi_raw_panel.csv'
df.to_csv(output_filename, index=False)
print(f"File saved: {output_filename}")

Targeting 46 countries with Max GDP > $35000
File saved: ../../data/wdi_raw_panel.csv
